# Stage1 冻结恢复评分与局部净动作价值

冻结两条独立 2/5bp 原生路径的 944 个交易日，严格同成本/同日期配对保存的恢复评分与到下一开盘共同重置的 Tiered-fee 优势金额。只做只读排序诊断和描述性区块区间；不修改 0.825 门、不训练、不新建账户。

In [1]:
import os,json,hashlib,pickle,tarfile,tempfile,subprocess,shlex
from pathlib import Path
import numpy as np,pandas as pd,qlib
from qlib.data import D
from qlib.data.dataset import DatasetH
from qlib.data.dataset.handler import DataHandlerLP
from qlib.workflow import R
from scipy.stats import spearmanr
from src.data.nas_market_data import archive_nas_recorder,_settings
root=Path.cwd();card=root/'research/stage1_recovery_score_net_value_diagnostic_20261023';plan=json.loads((card/'plan.json').read_text())
def sha(p):return hashlib.sha256(Path(p).read_bytes()).hexdigest()
assert plan['frozen'] and not (card/'run_state.json').exists()
for spec in (plan['score_source'],plan['advantage_source']):
 assert sha(root/spec['summary'])==spec['summary_sha256']
 s=json.loads((root/spec['summary']).read_text());assert s['recorder_id']==spec['recorder_id'] and s['nas_evidence']['sha256']
 assert sha(root/s['recorder_uri']/s['experiment_id']/s['recorder_id']/'artifacts'/spec['artifact'])==spec['artifact_sha256']
state={'status':'started','source_Recorder_object_loads':0,'reused_outcome_label_objects':0,'model_fits':0,'native_accounts':0};(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n')
os.environ['MLFLOW_ALLOW_FILE_STORE']='true';qlib.init(provider_uri=str(root/plan['provider']),region='us',kernels=1,expression_cache=None,dataset_cache=None)
a=json.loads((root/plan['score_source']['summary']).read_text());b=json.loads((root/plan['advantage_source']['summary']).read_text())
with R.uri_context(a['recorder_uri']):score_paths=R.get_recorder(recorder_id=a['recorder_id'],experiment_id=a['experiment_id']).load_object(plan['score_source']['artifact'])
state['source_Recorder_object_loads']+=1
with R.uri_context(b['recorder_uri']):labels=R.get_recorder(recorder_id=b['recorder_id'],experiment_id=b['experiment_id']).load_object(plan['advantage_source']['artifact'])
state['source_Recorder_object_loads']+=1;state['reused_outcome_label_objects']=1;(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n')
assert set(score_paths)=={'2','5'} and len(labels)==704
scores=[]
for key,path in sorted(score_paths.items()):
 assert len(path)==944 and path.trade_date.is_unique
 copy=path.copy();copy['slippage_bps']=int(key);scores.append(copy)
scores=pd.concat(scores,ignore_index=True);scores['trade_date']=pd.to_datetime(scores.trade_date);labels=labels.copy();labels['trade_date']=pd.to_datetime(labels.trade_date)
assert not scores.duplicated(['slippage_bps','trade_date']).any() and not labels.duplicated(['slippage_bps','trade_date']).any()
calendar=pd.DatetimeIndex(D.calendar(start_time=plan['cohorts']['trade_dates'][0],end_time=plan['cohorts']['trade_dates'][1],freq='day'))
assert len(calendar)==944
for cost in (2,5):assert set(scores.loc[scores.slippage_bps==cost,'trade_date'])==set(calendar)
joined=scores.merge(labels,on=['slippage_bps','trade_date'],how='left',validate='one_to_one')
assert len(joined)==1888
joined['mature_label']=joined.small_advantage.notna()&joined.label_available_at.notna()
joined['score_eligible']=joined.alpha_ready.astype(bool)&joined.cap_ready.astype(bool)&np.isfinite(joined.recovery_probability.to_numpy(float))
scored=joined[joined.score_eligible&joined.mature_label].copy()
assert [int(joined.loc[joined.slippage_bps==c,'score_eligible'].sum()) for c in (2,5)]==plan['cohorts']['route_decidable_each']
assert [len(scored[scored.slippage_bps==c]) for c in (2,5)]==plan['cohorts']['mature_same_state_labels_each']
assert scored.large_advantage.notna().all() and (pd.to_datetime(scored.label_available_at,utc=True)>pd.to_datetime(scored.trade_date,utc=True)).all()
frame=joined.set_index(['trade_date','slippage_bps'])[['recovery_probability','small_advantage','large_advantage','score_eligible','mature_label']]
frame.index=pd.MultiIndex.from_arrays([frame.index.get_level_values(0),['SOXL_SOXX_CASH_'+str(c)+'BP' for c in frame.index.get_level_values(1)]],names=['datetime','instrument'])
static=pd.concat({'feature':frame[['recovery_probability']],'label':frame[['small_advantage','large_advantage']],'meta':frame[['score_eligible','mature_label']]},axis=1)
dataset=DatasetH(handler=DataHandlerLP.from_df(static),segments={'held':tuple(plan['cohorts']['trade_dates'])})
assert len(dataset.prepare('held',col_set=['feature','label','meta']))==1888
rng=np.random.default_rng(20261023);statistics=[];annual=[];replicates={}
for cost in (2,5):
 x=scored[scored.slippage_bps==cost].sort_values('trade_date').reset_index(drop=True);score=x.recovery_probability.to_numpy(float);small=x.small_advantage.to_numpy(float);large=x.large_advantage.to_numpy(float);assert len(x)>200
 rho_small=float(spearmanr(score,small).statistic);rho_large=float(spearmanr(score,large).statistic)
 reps=[];n=len(x)
 for _ in range(2000):
  indices=np.empty(n,dtype=int);indices[0]=rng.integers(n)
  for j in range(1,n):indices[j]=rng.integers(n) if rng.random()<1/63 else (indices[j-1]+1)%n
  reps.append(float(spearmanr(score[indices],small[indices]).statistic))
 reps=np.array(reps);assert np.isfinite(reps).all();replicates[str(cost)]=reps
 lo,hi=np.quantile(reps,[.025,.975]);statistics.append({'slippage_bps':cost,'mature_route_decidable_rows':n,'score_min':float(score.min()),'score_max':float(score.max()),'small_advantage_mean_bp':float(small.mean()*10000),'large_advantage_mean_bp':float(large.mean()*10000),'score_vs_small_advantage_spearman':rho_small,'score_vs_large_advantage_spearman':rho_large,'small_advantage_spearman_descriptive_95pct_block_interval':[float(lo),float(hi)]})
 for year,g in x.groupby(x.trade_date.dt.year):
  annual.append({'slippage_bps':cost,'year':int(year),'rows':len(g),'mean_recovery_score':float(g.recovery_probability.mean()),'mean_small_advantage_bp':float(g.small_advantage.mean()*10000),'score_small_advantage_spearman':float(spearmanr(g.recovery_probability,g.small_advantage).statistic) if len(g)>10 else None})
audit={'status':'frozen_recovery_score_same_state_net_advantage_magnitude_diagnostic_only','plan_sha256':sha(card/'plan.json'),'sessions_per_cost':944,'route_decidable_per_cost':plan['cohorts']['route_decidable_each'],'mature_same_state_labels_per_cost':plan['cohorts']['mature_same_state_labels_each'],'statistics':statistics,'annual':annual,'model_fits':0,'native_accounts':0,'new_market_requests':0,'reused_outcome_label_objects':1,'local_labels_are_full_strategy_value':False,'historical_independence':False}
(card/'source_audit.json').write_text(json.dumps(audit,indent=2)+'\n')
dataset.config(dump_all=True,recursive=True);assert len(pickle.loads(pickle.dumps(dataset)).prepare('held',col_set=['feature','label','meta']))==1888
tracking=card/'qlib_recorder'
with R.start(experiment_name='Stage1_recovery_score_same_state_advantage_diagnostic',uri=str(tracking)):
 rec=R.get_recorder();rec.save_objects(artifact_path='inputs',plan=plan,source_hashes={'score':plan['score_source']['artifact_sha256'],'labels':plan['advantage_source']['artifact_sha256']})
 rec.save_objects(artifact_path='analysis',matched_daily=joined,scored_daily=scored,statistics=statistics,annual=annual,bootstrap_replicates=replicates,source_audit=audit)
 rec.save_objects(artifact_path='metadata',dataset=dataset)
 rec.log_metrics(mature_scored_rows=len(scored),score_spearman_2bp=statistics[0]['score_vs_small_advantage_spearman'],score_spearman_5bp=statistics[1]['score_vs_small_advantage_spearman'],model_fits=0,native_accounts=0)
 recorder_id=rec.id;experiment_id=str(rec.experiment_id)
with R.uri_context(str(tracking)):
 checked=R.get_recorder(recorder_id=recorder_id,experiment_id=experiment_id);assert len(checked.load_object('analysis/scored_daily'))==702 and len(checked.load_object('metadata/dataset').prepare('held',col_set=['feature','label','meta']))==1888
settings=_settings(root);binding=card/'nas_binding.json';binding.write_text(json.dumps({'schema_version':'recorder_evidence_binding_v1','authority':'yanxuan_nas','source_id':'Stage1_recovery_score_same_state_advantage_diagnostic','ssh':settings['ssh'],'source_plan':'plan.json'},indent=2)+'\n')
evidence=archive_nas_recorder(binding,workspace_root=root,card=card,recorder_id=recorder_id,experiment_id=experiment_id)
with tempfile.TemporaryDirectory(prefix='recovery-score-value-restore-') as temp:
 archive=Path(temp)/'recorder.tar.gz'
 with archive.open('wb') as out:run=subprocess.run(['ssh','-p',str(settings['ssh']['port']),'-o','BatchMode=yes',settings['ssh']['host'],'cat '+shlex.quote(evidence['nas_path'])],stdout=out,stderr=subprocess.PIPE,timeout=90)
 assert run.returncode==0 and sha(archive)==evidence['sha256']
 with tarfile.open(archive,'r:gz') as source:names=[m.name for m in source.getmembers() if m.isfile()]
 assert any(name.endswith('/artifacts/metadata/dataset') for name in names)
summary={'experiment':card.name,'decision':'diagnostic_only_recovery_score_local_net_advantage_rank_no_policy_change','objective_achieved':False,'plan':'plan.json','source_audit':'source_audit.json','model_fits':0,'native_accounts':0,'recorder_id':recorder_id,'experiment_id':experiment_id,'recorder_uri':str(tracking),'nas_evidence':evidence,'nas_readback':{'sha256':evidence['sha256'],'verified':True},'artifact_policy':{'policy':'slim_card_recorder_first','workflow_config':'workflow.yaml','recorder_only_artifacts':['inputs/*','analysis/*','metadata/*']}}
(card/'summary.json').write_text(json.dumps(summary,indent=2)+'\n')
state['status']='completed';state['recorder_id']=recorder_id;state['nas_archive_sha256']=evidence['sha256'];(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n')
print({'statistics':statistics,'annual':annual,'recorder_id':recorder_id,'nas_sha256':evidence['sha256']})


{'statistics': [{'slippage_bps': 2, 'mature_route_decidable_rows': 227, 'score_min': 0.08675201036645579, 'score_max': 0.7146916820764788, 'small_advantage_mean_bp': -2.0894861456824088, 'large_advantage_mean_bp': -4.099546103236243, 'score_vs_small_advantage_spearman': 0.05949188397292402, 'score_vs_large_advantage_spearman': 0.07811740180438233, 'small_advantage_spearman_descriptive_95pct_block_interval': [-0.02812092641868604, 0.1312179670957194]}, {'slippage_bps': 5, 'mature_route_decidable_rows': 475, 'score_min': 0.11630888551022195, 'score_max': 0.7151089522945079, 'small_advantage_mean_bp': 4.814875382166414, 'large_advantage_mean_bp': 14.17252222610412, 'score_vs_small_advantage_spearman': -0.012089643319019813, 'score_vs_large_advantage_spearman': 0.01345926245248266, 'small_advantage_spearman_descriptive_95pct_block_interval': [-0.0874206957266907, 0.05771307982125254]}], 'annual': [{'slippage_bps': 2, 'year': 2023, 'rows': 47, 'mean_recovery_score': 0.43915392447881507, 'me

## 中文结论

冻结恢复评分对同状态 0.6 beta 小动作的 Tiered 费后局部优势金额排序：2bp 227 日 Spearman 0.0595，描述性区块区间 [-0.0281,0.1312]；5bp 475 日为 -0.0121，区间 [-0.0874,0.0577]。均未显示可靠正排序，不下调原 0.825 门槛。此标签只到次日共同重置，不是完整账户价值。Qlib DatasetH/Recorder `fe947966067f452bb61a98fc8763e7de` 与 NAS SHA-256 `b2a60b19639a9a2fa10fce6c298bb67febc08fd731f29d2d6e8696332acba56d` 回读通过；0 拟合、0 新账户。